# 00 — Project Setup & Dataset

## AI-Based Predictive Maintenance & Equipment Failure Detection System

**Purpose:** Set up the project environment, inspect the dataset, and provide a reproducible synthetic industrial dataset for this college project.

> **Dataset note:** This project includes a reproducible synthetic dataset designed for academic demonstration. It contains the exact project input fields requested: temperature, pressure, vibration, rotational speed, voltage, current, operating hours, tool wear, machine type, and maintenance history. It is not real factory telemetry.

### Target
- `Failure = 0` → Normal
- `Failure = 1` → Failure Risk

### Important
For a real industrial deployment, sensor calibration, time-series validation, failure definitions, maintenance logs, and domain-expert validation would be required.

In [1]:
# Core imports
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)

Project root: D:\Python_Programming\Python_Git\Machine-Learning\Projects\1.AI-Based Predictive Maintenance & Equipment Failure Detection System
Raw data folder: D:\Python_Programming\Python_Git\Machine-Learning\Projects\1.AI-Based Predictive Maintenance & Equipment Failure Detection System\data\raw


## Dataset columns

| Feature | Meaning |
|---|---|
| Temperature_C | Machine/environment temperature in °C |
| Pressure_bar | Operating pressure in bar |
| Vibration_mm_s | Vibration level |
| Rotational_Speed_rpm | Rotational speed |
| Voltage_V | Supply voltage |
| Current_A | Electrical current |
| Operating_Hours | Cumulative operating hours |
| Tool_Wear_Min | Tool wear in minutes |
| Machine_Type | Machine category: L/M/H |
| Maintenance_History_Count | Number of previous maintenance events |
| Failure | Target variable |

In [2]:
# Load the included dataset
data_path = RAW_DIR / "dataset.csv"

if not data_path.exists():
    raise FileNotFoundError(
        f"{data_path} was not found. Keep the project folder structure unchanged."
    )

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
display(df.head())
display(df.info())

Dataset shape: (12000, 11)


,Temperature_C,Pressure_bar,Vibration_mm_s,Rotational_Speed_rpm,Voltage_V,Current_A,Operating_Hours,Tool_Wear_Min,Machine_Type,Maintenance_History_Count,Failure
0,36.67,7.91,1.94,1997,224.55,18.93,4370,117,M,2,0
1,27.73,8.13,3.77,1804,218.02,17.32,8183,84,L,1,1
2,32.57,8.93,2.40,1425,240.15,16.87,4021,78,H,1,0
3,31.17,8.90,3.87,1944,226.64,20.34,4853,237,M,2,0
4,29.08,7.01,5.15,1566,224.83,13.30,7184,41,L,1,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Temperature_C              12000 non-null  float64
 1   Pressure_bar               12000 non-null  float64
 2   Vibration_mm_s             12000 non-null  float64
 3   Rotational_Speed_rpm       12000 non-null  int64  
 4   Voltage_V                  12000 non-null  float64
 5   Current_A                  12000 non-null  float64
 6   Operating_Hours            12000 non-null  int64  
 7   Tool_Wear_Min              12000 non-null  int64  
 8   Machine_Type               12000 non-null  object 
 9   Maintenance_History_Count  12000 non-null  int64  
 10  Failure                    12000 non-null  int64  
dtypes: float64(5), int64(5), object(1)
memory usage: 1.0+ MB


None

In [3]:
# Basic quality checks
print("Missing values:")
display(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
display(df["Failure"].value_counts())
display(df["Failure"].value_counts(normalize=True).rename("percentage"))

Missing values:


Temperature_C                0
Pressure_bar                 0
Vibration_mm_s               0
Rotational_Speed_rpm         0
Voltage_V                    0
Current_A                    0
Operating_Hours              0
Tool_Wear_Min                0
Machine_Type                 0
Maintenance_History_Count    0
Failure                      0
dtype: int64


Duplicate rows: 0

Target distribution:


Failure
0    10433
1     1567
Name: count, dtype: int64

Failure
0    0.869417
1    0.130583
Name: percentage, dtype: float64

## Optional: Regenerate the dataset

The next cell can regenerate the included dataset using a fixed random seed. **Do not run it unless you intentionally want to recreate the data.**

In [4]:
# Optional synthetic-data generator
# Run only if you want to recreate data/raw/dataset.csv.

rng = np.random.default_rng(RANDOM_STATE)
n = 12000

machine_type = rng.choice(["L", "M", "H"], size=n, p=[0.50, 0.30, 0.20])
temperature = np.clip(rng.normal(35, 6, n), 18, 58)
pressure = np.clip(rng.normal(7.5, 1.6, n), 3, 13)
vibration = np.clip(rng.gamma(2.2, 1.4, n), 0.4, 12)
speed = np.clip(rng.normal(1800, 380, n), 800, 3000)
voltage = np.clip(rng.normal(230, 8, n), 205, 255)
current = np.clip(
    6 + (speed / 1800) * 8 + (pressure / 8) * 3 + rng.normal(0, 2, n),
    2, 30
)
operating_hours = np.clip(rng.gamma(3.0, 1700, n), 50, 15000)
tool_wear = rng.integers(0, 251, n)
maintenance = np.clip(rng.poisson(2.2, n), 0, 9)

risk_signal = (
    0.10 * (temperature - 35)
    + 0.55 * (vibration - 3)
    + 0.35 * (pressure - 8)
    + 0.0020 * (operating_hours - 5000)
    + 0.018 * (tool_wear - 120)
    + 0.12 * (current - 14)
    + 0.0005 * np.abs(speed - 1800)
    - 0.10 * (maintenance - 2)
)

risk_signal += (
    1.2 * (temperature > 47)
    + 1.0 * (vibration > 7)
    + 0.9 * (tool_wear > 210)
    + 0.8 * (operating_hours > 10000)
    + rng.normal(0, 1.8, n)
)

failure_probability = 1 / (1 + np.exp(-(risk_signal - 8.5)))
failure = rng.binomial(1, failure_probability)

generated_df = pd.DataFrame({
    "Temperature_C": temperature.round(2),
    "Pressure_bar": pressure.round(2),
    "Vibration_mm_s": vibration.round(2),
    "Rotational_Speed_rpm": speed.round(0).astype(int),
    "Voltage_V": voltage.round(2),
    "Current_A": current.round(2),
    "Operating_Hours": operating_hours.round(0).astype(int),
    "Tool_Wear_Min": tool_wear,
    "Machine_Type": machine_type,
    "Maintenance_History_Count": maintenance,
    "Failure": failure
})

generated_df.to_csv(data_path, index=False)
print("Regenerated:", data_path)
print("Shape:", generated_df.shape)
print("Failure rate:", generated_df["Failure"].mean())

Regenerated: D:\Python_Programming\Python_Git\Machine-Learning\Projects\1.AI-Based Predictive Maintenance & Equipment Failure Detection System\data\raw\dataset.csv
Shape: (12000, 11)
Failure rate: 0.13058333333333333
